# Graph Isomorphism Network (GIN) on MUTAG

Graph Classification on MUTAG (TUDataset): Maximally expressive Weisfeiler-Lehman graph classification on MUTAG molecules. This notebook implements the approach with `GINConv` inside a `K3GIN` model, trained with the Adam optimizer for 20 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GINConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset
from k3_node.loader import DataLoader

title = "Graph Isomorphism Network (GIN) on MUTAG"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
dataset = TUDataset(root="./data/MUTAG", name="MUTAG")
train_batch_size = 32
# drop_last=True keeps every training batch at a fixed size. Keras's
# fit()/train_on_batch() runs one internal forward pass with constant-filled
# placeholder tensors to validate the loss pipeline; if the graph count in a
# batch were only known from real data (e.g. inferred as `batch.max()+1`),
# that placeholder pass would infer a different (wrong) output size and the
# validation pass would raise a spurious shape-mismatch error. With a fixed
# batch size we instead pass a static `size` to global_add_pool.
train_loader = DataLoader(dataset[:150], batch_size=train_batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset[150:], batch_size=32)

in_channels = dataset.num_features
num_classes = dataset.num_classes

# 2. GIN Model Definition
class K3GIN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, batch_size):
        super().__init__()
        nn1 = keras.Sequential([layers.Dense(hidden_channels, activation="relu"), layers.Dense(hidden_channels)])
        self.conv1 = k3_layers.GINConv(nn1, train_eps=True)
        nn2 = keras.Sequential([layers.Dense(hidden_channels, activation="relu"), layers.Dense(hidden_channels)])
        self.conv2 = k3_layers.GINConv(nn2, train_eps=True)
        self.lin = layers.Dense(out_channels)
        self.batch_size = batch_size

    def call(self, inputs):
        x, edge_index, batch = inputs["x"], inputs["edge_index"], inputs.get("batch", None)
        x = ops.relu(self.conv1(x, edge_index))
        x = ops.relu(self.conv2(x, edge_index))
        out = k3_layers.global_add_pool(x, batch, size=self.batch_size)
        return self.lin(out)

k3_model = K3GIN(in_channels, 64, num_classes, batch_size=train_batch_size)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Generator & Training
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            inputs = {
                "x": to_np(batch.x, dtype=np.float32),
                "edge_index": to_np(batch.edge_index, dtype=np.int64),
                "batch": to_np(batch.batch, dtype=np.int64) if hasattr(batch, "batch") else None,
            }
            y = to_np(batch.y, dtype=np.int64).reshape(-1)
            yield inputs, y

print(f"Training K3-Node GIN on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=20,
    verbose=1,
)

print("\n✓ K3-Node GIN execution completed successfully!")